# 🧪 W2-D3 概念实验：GPT 与 BERT 架构对比

> 配套阅读：`第2周-Day3-GPT与BERT架构对比.md`（架构讲解与误区澄清在那边）
> 这个 notebook 用可执行实验回答四个问题：
> **掩码差在哪 / 信息能流多远 / 参数账怎么算 / 为什么生成必须因果掩码**
>
> 实验环境：纯 numpy + matplotlib。

## 实验 1：两种掩码 —— 一字之差，世界观之别

BERT：全 1 矩阵（双向，每个词看到全句）。
GPT：下三角矩阵（因果，每个词只看到自己和左边）。
打印出来直观看一眼。

In [ ]:
import numpy as np

n = 6
gpt_mask = np.tril(np.ones((n, n), dtype=int))    # 因果：只看左边（含自己）
bert_mask = np.ones((n, n), dtype=int)            # 双向：全可见

print("GPT 因果掩码（1=可见）：")
for i, row in enumerate(gpt_mask):
    print(f"  位置{i}: " + " ".join(map(str, row)) + f"   可见 {row.sum()} 个词")
print("BERT 双向掩码：全 1，每个位置都能看全句")
print("\n→ 同一套注意力公式，掩码不同 = 信息通路不同 = 任务能力不同")

## 实验 2：信息可达性 —— "它"永远看不到"猫"

句子 `它 累 了 ， 因为 猫 追 了 半天`：代词"它"(位置0)指代"猫"(位置5)，**指代对象在右边**。
用布尔矩阵幂模拟"堆 L 层后谁能看到谁"：GPT 堆多少层，位置 0 的可达集都不包含位置 5；
BERT 一层就全通。这就是"生成模型天生不适合做双向理解"的结构性原因。

In [ ]:
def reachable(mask, L):
    """堆 L 层自注意力后，位置 i 的信息可达集（1 层 = 直接可见）"""
    R = mask.astype(int)
    reach = R.copy()
    for _ in range(L - 1):
        reach = ((reach @ R) > 0).astype(int)   # 可达的可达 = 传递
    return reach

sent = ["它", "累", "了", "，", "因为", "猫"]
for L in [1, 2, 3, 12]:
    g = reachable(gpt_mask, L)
    b = reachable(bert_mask, L)
    print(f"堆 {L:>2} 层 | GPT: 位置0可见 {g[0].sum()} 个词（含'猫'？{'是' if g[0,5] else '否'}）"
          f" | BERT: 位置0可见 {b[0].sum()} 个词（含'猫'？{'是' if b[0,5] else '否'}）")

print("\n→ GPT 的可达集永远是下三角的子集：右边的词永远进不来（因果性是硬约束）")
print("→ BERT 一层全通，做理解任务（分类/NER/检索）信息效率更高")

## 实验 3：同一组 Q/K，两种注意力热力图

随机初始化的 6 词 × 16 维，分别用双向/因果掩码算权重矩阵。
GPT 版右上三角全黑（未来词权重=0），BERT 版整图有值。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

rng = np.random.default_rng(3)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

n, d = 6, 16
X = rng.normal(size=(n, d))
Wq = rng.normal(scale=0.6, size=(d, d)); Wk = rng.normal(scale=0.6, size=(d, d))
Q, K = X @ Wq, X @ Wk
scores = Q @ K.T / np.sqrt(d)

W_bert = softmax(scores, axis=1)
s_gpt = np.where(gpt_mask == 1, scores, -1e9)   # 因果掩码在 softmax 之前生效
W_gpt = softmax(s_gpt, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, mat, t in [(axes[0], W_bert, "BERT：双向（全可见）"),
                   (axes[1], W_gpt, "GPT：因果（只看左边）")]:
    im = ax.imshow(mat, cmap="YlOrRd", vmin=0)
    ax.set_xticks(range(n), sent, rotation=30); ax.set_yticks(range(n), sent)
    ax.set_xlabel("被看的词"); ax.set_ylabel("查询的词"); ax.set_title(t)
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle("同一组 Q/K，掩码决定信息通路")
plt.tight_layout(); plt.show()

print("位置0（'它'）在 GPT 下的权重:", dict(zip(sent, W_gpt[0].round(3))), " ← 对'猫'为 0")

## 实验 4：参数量公式 —— 12·d²·L + vocab·d

每层 ≈ 4d²（QKVO 投影）+ 8d²（FFN 两层）= 12d²，加词嵌入 vocab·d。
用公开模型配置验证公式：BERT-base / GPT-2 small / GPT-3 175B，误差都在 5% 内——
**说明 Encoder 和 Decoder 的每一层参数结构其实一样，差别只在掩码和训练目标**。

In [ ]:
def params_B(d, L, vocab):
    per_layer = 4*d*d + 2*d*(4*d)          # 注意力 4d² + FFN 8d²
    return (L * per_layer + vocab * d) / 1e9

models = [
    ("BERT-base",  768,  12, 30522, 0.110),
    ("GPT-2 small",768,  12, 50257, 0.117),
    ("GPT-3 175B", 12288,96, 50257, 175.0),
]
print(f"{'模型':<12}{'d_model':>8}{'层数':>5}{'词表':>8}  {'公式计算':>10}{'公开值':>10}{'误差':>7}")
for name, d, L, v, pub in models:
    est = params_B(d, L, v)
    print(f"{name:<12}{d:>8}{L:>5}{v:>8}  {est:>9.1f}B{pub:>9.1f}B{abs(est-pub)/pub:>6.1%}")
print("\n→ GPT-3: 96 层 × 12 × 12288² ≈ 174B，公式几乎精确命中")
print("→ 参数量由 d 和 L 决定；GPT/BERT 之争不在参数，在掩码 + 训练目标（下一词 vs 完形填空）")

## 实验 5：为什么生成必须因果掩码 —— 防"作弊"

训练目标是"用位置 i 预测词 i+1"。如果不加掩码，位置 i 的注意力可以给**未来的答案**分配权重：
分数最高的 key 恰好是下一个词时，模型直接抄答案，训练指标虚高、推理时崩溃。
因果掩码保证"考卷不泄底"，训练（并行）与推理（逐词生成）行为一致。

In [ ]:
n5, d5 = 5, 8
X5 = rng.normal(size=(n5, d5))
Wq5 = rng.normal(scale=0.6, size=(d5, d5)); Wk5 = rng.normal(scale=0.6, size=(d5, d5))
Q5, K5 = X5 @ Wq5, X5 @ Wk5

# 故意把位置 3（= 位置 2 要预测的"未来答案"）的 key 与位置 2 的 query 对齐
Q5[2] = 2.0 * K5[3]

s5 = Q5 @ K5.T / np.sqrt(d5)
w_free = softmax(s5, axis=1)
w_causal = softmax(np.where(np.tril(np.ones((n5, n5), dtype=int)) == 1, s5, -1e9), axis=1)

print("不加掩码，位置 2 的注意力:", w_free[2].round(3), " ← 63%+ 权重给了位置 3（未来=答案）")
print("加因果掩码，位置 2 的注意力:", w_causal[2].round(3), " ← 未来被屏蔽，只能用左边的信息")
print("\n→ 生成式训练 = 闭卷考试：因果掩码挡住答案，模型才学得到真实预测能力")

## 结论

| 维度 | BERT（Encoder-only） | GPT（Decoder-only） |
|---|---|---|
| 掩码 | 全可见（实验 1） | 下三角（实验 1） |
| 信息流 | 1 层全通（实验 2） | 永远左向（实验 2） |
| 每层参数 | 12d² + FFN 8d²，两者相同（实验 4） | 同左 |
| 训练目标 | 完形填空（双向理解） | 下一词预测（必须因果掩码，实验 5） |

→ 深入阅读：同目录 `.md` 版本 2.4 节（为什么大模型时代 GPT 架构胜出）